# 10 - Cost Regression: Predicting the Dollar Amount

the classifiers (notebook 05) answer *who* -- a binary `high_cost` flag. This notebook answers *how much* -- a continuous prediction of each member's actual 2010 spend. The two are complementary, not competing: the classifier stays the primary targeting mechanism (a care-management team acts on a flag, not a fractional one), while this adds two things the classifier structurally cannot give you -- a defensible cohort-level budget forecast (sum every member's predicted spend), and a way to rank *within* the flagged group by predicted magnitude once even the top tier exceeds capacity.\n\n**one genuine simplification vs. notebook 05:** there's no percentile threshold to freeze here -- the target is the dollar amount itself, so the whole train-then-freeze-the-boundary discipline that mattered for `high_cost` simply doesn't apply to a continuous target.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import lightgbm as lgb
from lightgbm import LGBMRegressor

with open("../configs/model_config.yaml") as f:
    config = yaml.safe_load(f)

FEATURE_COLS = [
    "inpatient_admits", "outpatient_visits", "carrier_claims",
    "inpatient_paid", "outpatient_paid", "carrier_paid",
    "has_alzheimers", "has_chf", "has_ckd", "has_cancer", "has_copd",
    "has_depression", "has_diabetes", "has_ihd", "has_osteoporosis",
    "has_ra_oa", "has_stroke_tia", "has_esrd",
    "age", "is_male", "race_black", "race_other", "race_hispanic",
    "rx_fill_count", "rx_total_cost", "rx_patient_paid",
    "other_payer_paid",
]
SKEWED_COLS = [
    "inpatient_paid", "outpatient_paid", "carrier_paid",
    "outpatient_visits", "carrier_claims",
    "rx_fill_count", "rx_total_cost", "rx_patient_paid",
    "other_payer_paid",
]


**target transform.** `total_paid_2010` is heavily right-skewed (median $1,200, mean $2,976, max $116,910 -- see notebook 04). Training squared-error regression directly on raw dollars would let a handful of $100k+ members dominate the loss. The standard fix, and the one used here first: train on `log1p(total_paid_2010)`, predict, then `expm1` back to dollars for every reported metric. A Tweedie-loss comparison (the actuarial-industry-standard for zero-inflated, right-skewed cost data) is run afterward against this baseline, same pattern as the earlier Optuna ablation -- try the standard approach, then a documented comparison against the fancier one.

In [ ]:
# same split as notebook 05 (same random_state, same stratify column) so the
# exact same members land in train/test -- lets these results sit directly
# alongside the classifier's without re-deriving anything
df = pd.read_parquet("../data/processed/features_labeled.parquet")
X = df[FEATURE_COLS]
spend = df["total_paid_2010"]

X_train, X_test, spend_train, spend_test = train_test_split(
    X, spend,
    test_size=config["split"]["test_size"],
    stratify=df["high_cost"] if config["split"]["stratify"] else None,
    random_state=config["random_seed"],
)

# negative values (real claim-adjustment artifacts, see notebook 04) are
# clipped to 0 before the log -- log1p of a value below -1 is undefined
y_train_log = np.log1p(spend_train.clip(lower=0))
y_test_log = np.log1p(spend_test.clip(lower=0))


In [ ]:
# score predictions on the real dollar scale (expm1 back), not the log scale
# -- RMSE/MAE in dollars is what a stakeholder can actually interpret
def evaluate_regression(model, X_test, spend_test_actual, log_scale=True):
    pred = model.predict(X_test)
    pred_dollars = np.expm1(pred) if log_scale else pred
    pred_dollars = np.clip(pred_dollars, a_min=0, a_max=None)  # spend can't be negative
    return {
        "rmse": round(float(np.sqrt(mean_squared_error(spend_test_actual, pred_dollars))), 2),
        "mae": round(float(mean_absolute_error(spend_test_actual, pred_dollars)), 2),
        "r2": round(float(r2_score(spend_test_actual, pred_dollars)), 4),
    }


In [ ]:
# linear regression baseline, log1p-transformed skewed features + log target
# -- plays the same role logistic regression does for the classifiers
def log1p_clipped(x):
    return np.log1p(np.clip(x, a_min=0, a_max=None))

log_transform = ColumnTransformer(
    [("log1p", FunctionTransformer(log1p_clipped), SKEWED_COLS)],
    remainder="passthrough",
)

linreg_pipeline = Pipeline([
    ("log_transform", log_transform),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
linreg_pipeline.fit(X_train, y_train_log)

linreg_metrics = evaluate_regression(linreg_pipeline, X_test, spend_test)
linreg_metrics


In [ ]:
# carve the same early-stopping validation slice as notebook 05
X_fit, X_earlystop_val, y_fit_log, y_val_log = train_test_split(
    X_train, y_train_log, test_size=0.2, random_state=config["random_seed"]
)


In [ ]:
# xgboost regressor, log1p target, early stopping picks the round count
# (identical discipline to the classifier: search on a validation slice,
# then refit on the full training partition at the winning round count)
xgb_config = config["xgboost"]
xgb_search = XGBRegressor(
    n_estimators=xgb_config["n_estimators"],
    max_depth=xgb_config["max_depth"],
    learning_rate=xgb_config["learning_rate"],
    subsample=xgb_config["subsample"],
    colsample_bytree=xgb_config["colsample_bytree"],
    eval_metric="rmse",
    random_state=config["random_seed"],
    early_stopping_rounds=20,
)
xgb_search.fit(X_fit, y_fit_log, eval_set=[(X_earlystop_val, y_val_log)], verbose=False)
print(f"xgboost early stopping selected {xgb_search.best_iteration} rounds (of {xgb_config['n_estimators']} max)")

xgb_reg = XGBRegressor(
    n_estimators=xgb_search.best_iteration,
    max_depth=xgb_config["max_depth"],
    learning_rate=xgb_config["learning_rate"],
    subsample=xgb_config["subsample"],
    colsample_bytree=xgb_config["colsample_bytree"],
    eval_metric="rmse",
    random_state=config["random_seed"],
)
xgb_reg.fit(X_train, y_train_log)

xgb_reg_metrics = evaluate_regression(xgb_reg, X_test, spend_test)
xgb_reg_metrics


In [ ]:
# lightgbm regressor -- metric="rmse" set on the CONSTRUCTOR, not just
# fit(), for the same reason notebook 05 needed metric="auc": lightgbm
# registers its own default eval metric alongside whatever fit() is given,
# and early stopping tracks whichever one comes first unless told otherwise
lgbm_config = config["lightgbm"]
lgbm_search = LGBMRegressor(
    n_estimators=lgbm_config["n_estimators"],
    max_depth=lgbm_config["max_depth"],
    num_leaves=lgbm_config["num_leaves"],
    learning_rate=lgbm_config["learning_rate"],
    subsample=lgbm_config["subsample"],
    colsample_bytree=lgbm_config["colsample_bytree"],
    random_state=config["random_seed"],
    verbose=-1,
    metric="rmse",
)
lgbm_search.fit(
    X_fit, y_fit_log, eval_set=[(X_earlystop_val, y_val_log)],
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
)
print(f"lightgbm early stopping selected {lgbm_search.best_iteration_} rounds (of {lgbm_config['n_estimators']} max)")

lgbm_reg = LGBMRegressor(
    n_estimators=lgbm_search.best_iteration_,
    max_depth=lgbm_config["max_depth"],
    num_leaves=lgbm_config["num_leaves"],
    learning_rate=lgbm_config["learning_rate"],
    subsample=lgbm_config["subsample"],
    colsample_bytree=lgbm_config["colsample_bytree"],
    random_state=config["random_seed"],
    verbose=-1,
)
lgbm_reg.fit(X_train, y_train_log)

lgbm_reg_metrics = evaluate_regression(lgbm_reg, X_test, spend_test)
lgbm_reg_metrics


In [ ]:
pd.DataFrame(
    [linreg_metrics, xgb_reg_metrics, lgbm_reg_metrics],
    index=["linear_regression", "xgboost", "lightgbm"],
)


**5-fold CV, training partition only** -- same discipline as notebook 05: the test set stays untouched, only used once at the very end.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config["random_seed"])
# StratifiedKFold needs a categorical column to stratify on -- reuse high_cost
# from the training rows purely to keep the fold composition comparable to
# notebook 05's CV, not because regression itself needs stratification
strat_labels = df.loc[X_train.index, "high_cost"]

def rmse_scorer(estimator, X, y_log):
    pred_dollars = np.clip(np.expm1(estimator.predict(X)), a_min=0, a_max=None)
    actual_dollars = np.expm1(y_log)
    return -np.sqrt(mean_squared_error(actual_dollars, pred_dollars))  # negative: higher=better for cross_validate

xgb_cv = cross_validate(xgb_reg, X_train, y_train_log, cv=cv.split(X_train, strat_labels), scoring=rmse_scorer)
lgbm_cv = cross_validate(lgbm_reg, X_train, y_train_log, cv=cv.split(X_train, strat_labels), scoring=rmse_scorer)
linreg_cv = cross_validate(linreg_pipeline, X_train, y_train_log, cv=cv.split(X_train, strat_labels), scoring=rmse_scorer)

pd.DataFrame({
    "linear_regression": [-linreg_cv["test_score"].mean(), linreg_cv["test_score"].std()],
    "xgboost": [-xgb_cv["test_score"].mean(), xgb_cv["test_score"].std()],
    "lightgbm": [-lgbm_cv["test_score"].mean(), lgbm_cv["test_score"].std()],
}, index=["cv_rmse_mean", "cv_rmse_std"])


**Tweedie comparison.** Tweedie regression is built specifically for data shaped like ours -- a spike of exact zeros (many members have $0 in some spend category) plus a continuous, right-skewed positive tail. It's the standard loss function in actuarial/insurance cost modeling for exactly this reason. Trained here directly on raw dollars, not the log1p target -- Tweedie's own distributional assumption is meant to handle the skew natively.

In [ ]:
xgb_tweedie = XGBRegressor(
    n_estimators=xgb_search.best_iteration,
    max_depth=xgb_config["max_depth"],
    learning_rate=xgb_config["learning_rate"],
    subsample=xgb_config["subsample"],
    colsample_bytree=xgb_config["colsample_bytree"],
    objective="reg:tweedie",
    tweedie_variance_power=1.5,
    random_state=config["random_seed"],
)
xgb_tweedie.fit(X_train, spend_train.clip(lower=0))
xgb_tweedie_metrics = evaluate_regression(xgb_tweedie, X_test, spend_test, log_scale=False)

lgbm_tweedie = LGBMRegressor(
    n_estimators=lgbm_search.best_iteration_,
    max_depth=lgbm_config["max_depth"],
    num_leaves=lgbm_config["num_leaves"],
    learning_rate=lgbm_config["learning_rate"],
    subsample=lgbm_config["subsample"],
    colsample_bytree=lgbm_config["colsample_bytree"],
    objective="tweedie",
    random_state=config["random_seed"],
    verbose=-1,
)
lgbm_tweedie.fit(X_train, spend_train.clip(lower=0))
lgbm_tweedie_metrics = evaluate_regression(lgbm_tweedie, X_test, spend_test, log_scale=False)

pd.DataFrame(
    [xgb_reg_metrics, xgb_tweedie_metrics, lgbm_reg_metrics, lgbm_tweedie_metrics],
    index=["xgboost (log1p target)", "xgboost (tweedie)", "lightgbm (log1p target)", "lightgbm (tweedie)"],
)


**Tweedie wins on R² -- and there's a second, more important reason to prefer it: retransformation bias.** A model trained on `log1p(spend)` and then `expm1()`-ed back to dollars is a well-known source of *systematic* underestimation of the true dollar-scale total, especially with spend this skewed -- exponentiating a prediction error that's symmetric on the log scale does not produce a prediction that's unbiased on the dollar scale. Tweedie avoids this entirely by training directly on raw dollars. The check below makes this concrete rather than theoretical.

In [ ]:
# does the retransformation bias actually show up in the aggregate total?
log1p_total = np.clip(np.expm1(lgbm_reg.predict(X_test)), a_min=0, a_max=None).sum()
tweedie_total = np.clip(lgbm_tweedie.predict(X_test), a_min=0, a_max=None).sum()
actual_total = spend_test.sum()

pd.DataFrame({
    "total_predicted": [log1p_total, tweedie_total],
    "pct_error_vs_actual": [
        (log1p_total - actual_total) / actual_total,
        (tweedie_total - actual_total) / actual_total,
    ],
}, index=["lightgbm (log1p target)", "lightgbm (tweedie)"]).style.format({"total_predicted": "${:,.0f}", "pct_error_vs_actual": "{:+.1%}"})


**picking the final model: Tweedie, based on both R² and the bias check above, not just whichever had the lowest RMSE.**

In [ ]:
candidates = {
    "linear_regression (log1p)": (linreg_pipeline, "log1p", linreg_metrics["r2"]),
    "xgboost (log1p)": (xgb_reg, "log1p", xgb_reg_metrics["r2"]),
    "lightgbm (log1p)": (lgbm_reg, "log1p", lgbm_reg_metrics["r2"]),
    "xgboost (tweedie)": (xgb_tweedie, "raw", xgb_tweedie_metrics["r2"]),
    "lightgbm (tweedie)": (lgbm_tweedie, "raw", lgbm_tweedie_metrics["r2"]),
}
best_name = max(candidates, key=lambda k: candidates[k][2])
best_model, best_transform, _ = candidates[best_name]
print(f"selected: {best_name}")

def predict_dollars(model, X, transform):
    pred = model.predict(X)
    pred = np.expm1(pred) if transform == "log1p" else pred
    return np.clip(pred, a_min=0, a_max=None)


**predicted vs. actual, on the test set.** A well-fit model's points should cluster around the diagonal; the spend of individual expensive members (top right) is where cost regression is hardest, same reason AP mattered more than AUC for the classifier -- most of the signal is concentrated in a small, hard-to-predict minority.

In [ ]:
pred_dollars = predict_dollars(best_model, X_test, best_transform)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(spend_test, pred_dollars, alpha=0.15, s=10)
lims = [0, max(spend_test.max(), pred_dollars.max())]
ax.plot(lims, lims, linestyle="--", color="grey", label="perfect prediction")
ax.set_xlabel("actual 2010 spend ($)")
ax.set_ylabel(f"predicted 2010 spend ($)")
ax.set_title(f"predicted vs. actual spend -- {best_name}")
ax.legend()
plt.tight_layout()
plt.show()


**the budget-forecast number the classifier can't give you.** Summing every member's predicted spend is exactly the kind of aggregate, dollar-denominated output a stakeholder can act on directly -- distinct from anything the classifier produces.

In [ ]:
actual_total = spend_test.sum()
predicted_total = pred_dollars.sum()
print(f"actual total 2010 spend (test set, {len(spend_test):,} members):    ${actual_total:,.0f}")
print(f"{best_name} predicted total 2010 spend (test set):        ${predicted_total:,.0f}")
print(f"difference: {(predicted_total - actual_total) / actual_total:+.1%}")


In [ ]:
# save all 5 candidate models + full-cohort predictions from the selected one
out_dir = Path("../data/processed/models")
out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(linreg_pipeline, out_dir / "linear_regression_cost.joblib")
joblib.dump(xgb_reg, out_dir / "xgboost_cost_log1p.joblib")
joblib.dump(lgbm_reg, out_dir / "lightgbm_cost_log1p.joblib")
joblib.dump(xgb_tweedie, out_dir / "xgboost_cost_tweedie.joblib")
joblib.dump(lgbm_tweedie, out_dir / "lightgbm_cost_tweedie.joblib")

full_predictions = predict_dollars(best_model, df[FEATURE_COLS], best_transform)
cost_predictions = pd.DataFrame({
    "desynpuf_id": df["desynpuf_id"],
    "predicted_2010_spend": full_predictions.round(2),
})
cost_predictions.to_parquet("../data/processed/member_cost_predictions.parquet", index=False)
print(f"wrote member_cost_predictions.parquet ({len(cost_predictions):,} members, model={best_name})")
